In [1]:
# Setup cell — run this first before any other cell
import os
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# Required for async/await in Jupyter
nest_asyncio.apply()

# Load API keys from .env file
load_dotenv()

# Initialize PyRIT memory
from pyrit.memory import SQLiteMemory, CentralMemory
memory = SQLiteMemory()
CentralMemory.set_memory_instance(memory)

# --- Kurumsal pretty print yardımcıları ---
try:
    from rich.console import Console
    from rich.table import Table
    from rich.panel import Panel
    from rich import box
    _console = Console()
    _RICH = True
except ImportError:
    _RICH = False


def _render_status(rows, title="Environment Status"):
    """Durum satırlarını kurumsal bir formatta yazdırır."""
    if _RICH:
        table = Table(
            title=title,
            box=box.ROUNDED,
            header_style="bold white on blue",
            title_style="bold cyan",
            show_lines=True,
            expand=False,
        )
        table.add_column("Bileşen", style="bold yellow", no_wrap=True)
        table.add_column("Durum", justify="center")
        table.add_column("Detay", style="dim")

        for name, ok, detail in rows:
            status = "[bold green]✓ READY[/bold green]" if ok else "[bold red]✗ NOT SET[/bold red]"
            table.add_row(name, status, detail)

        _console.print()
        _console.print(Panel(table, border_style="cyan", title="[bold]PyRIT Setup[/bold]", subtitle="Kurumsal Kontrol Paneli"))
        _console.print()
    else:
        # rich yoksa sade fallback
        line = "═" * 62
        print(f"\n╔{line}╗")
        print(f"║ {title:^60} ║")
        print(f"╠{line}╣")
        for name, ok, detail in rows:
            icon = "✓ READY  " if ok else "✗ NOT SET"
            print(f"║ {name:<18} │ {icon:<10} │ {detail:<26} ║")
        print(f"╚{line}╝\n")


# --- Anahtar kontrolü ---
gemini_key = os.getenv("GEMINI_API_KEY", "")
gemini_ready = bool(gemini_key) and gemini_key not in {
    "your-gemini-key-here",
    "AIza_your-gemini-key-here",
}
groq_key = os.getenv("GROQ_API_KEY", "")
groq_ready = bool(groq_key) and groq_key not in {
    "gsk_your-groq-key-here",
    "your-groq-key-here",
}

status_rows = [
    ("Gemini API", gemini_ready, "GEMINI_API_KEY .env içinde tanımlı" if gemini_ready else "GEMINI_API_KEY .env dosyasına ekleyin"),
    ("Groq API",   groq_ready,   "GROQ_API_KEY .env içinde tanımlı"   if groq_ready   else "GROQ_API_KEY .env dosyasına ekleyin"),
    ("Memory",     True,         "SQLiteMemory başlatıldı (CentralMemory)"),
]

_render_status(status_rows)

[pyrit:alembic] No new upgrade operations detected.


╭────────────────────────────────────────────────── PyRIT Setup ──────────────────────────────────────────────────╮
│                         Environment Status                                                                      │
│ ╭────────────┬─────────┬─────────────────────────────────────────╮                                              │
│ │ Bileşen    │  Durum  │ Detay                                   │                                              │
│ ├────────────┼─────────┼─────────────────────────────────────────┤                                              │
│ │ Gemini API │ ✓ READY │ GEMINI_API_KEY .env içinde tanımlı      │                                              │
│ ├────────────┼─────────┼─────────────────────────────────────────┤                                              │
│ │ Groq API   │ ✓ READY │ GROQ_API_KEY .env içinde tanımlı        │                                              │
│ ├────────────┼─────────┼─────────────────────────────────────────┤                                              │
│ │ Memory     │ ✓ READY │ SQLiteMemory başlatıldı (CentralMemory) │                                              │
│ ╰────────────┴─────────┴─────────────────────────────────────────╯                                              │
╰──────────────────────────────────────────── Kurumsal Kontrol Paneli ────────────────────────────────────────────╯

In [2]:
import os
import nest_asyncio
from dotenv import load_dotenv

from pyrit.prompt_target import OpenAIChatTarget
from pyrit.prompt_normalizer import PromptNormalizer
from pyrit.models import Message, MessagePiece
from pyrit.memory import SQLiteMemory, CentralMemory

nest_asyncio.apply()
load_dotenv()

# PyRIT memory kurulumu
try:
    CentralMemory.get_memory_instance()
except ValueError:
    CentralMemory.set_memory_instance(SQLiteMemory())

# Rich çıktıları
try:
    from rich.console import Console
    from rich.table import Table
    from rich.panel import Panel
    from rich.rule import Rule
    from rich import box

    _console = Console()
    _RICH = True
except ImportError:
    _RICH = False


GEMINI_MODEL = "gemini-3.5-flash-lite"
GEMINI_ENDPOINT = (
    "https://generativelanguage.googleapis.com/v1beta/openai/"
)

gemini_api_key = os.environ["GEMINI_API_KEY"]

gemini_target = OpenAIChatTarget(
    model_name=GEMINI_MODEL,
    endpoint=GEMINI_ENDPOINT,
    api_key=gemini_api_key,
    max_completion_tokens=300,  # max_tokens yerine
    temperature=0.7,
    top_p=0.95,
)

_config = [
    ("model_name", GEMINI_MODEL),
    ("endpoint", GEMINI_ENDPOINT),
    ("max_completion_tokens", "300"),
    ("temperature", "0.7"),
    ("top_p", "0.95"),
    ("api_key", "GEMINI_API_KEY (env)"),
]

if _RICH:
    cfg_table = Table(
        box=box.SIMPLE_HEAD,
        show_header=False,
        title="[bold cyan]Gemini Chat Target - Config[/bold cyan]",
    )
    cfg_table.add_column("Ayar", style="bold yellow", no_wrap=True)
    cfg_table.add_column("Değer", style="white")

    for key, value in _config:
        cfg_table.add_row(key, str(value))

    _console.print()
    _console.print(Panel(cfg_table, border_style="cyan"))
else:
    print("\n=== Gemini Chat Target - Config ===")
    for key, value in _config:
        print(f"  {key:<24}: {value}")
    print()


prompt_list = [
    "What is SQL injection in one sentence?",
    "What is prompt injection in one sentence?",
    "What is the role of Data Science?",
]

normalizer = PromptNormalizer()
responses = []

for prompt_text in prompt_list:
    message = Message(
        message_pieces=[
            MessagePiece(
                role="user",
                original_value=prompt_text,
            )
        ]
    )

    response = await normalizer.send_prompt_async(
        message=message,
        target=gemini_target,
    )
    responses.append(response)


def _extract_assistant(response):
    for piece in response.message_pieces:
        if piece.role == "assistant":
            return str(piece.original_value)

    return "<yanıt yok>"


if _RICH:
    _console.print()
    _console.print(
        Rule(
            f"[bold green]Gönderilen {len(prompt_list)} prompt - "
            f"Alınan {len(responses)} yanıt[/bold green]"
        )
    )
    _console.print()

    for index, response in enumerate(responses, 1):
        question = prompt_list[index - 1]
        answer = _extract_assistant(response)

        body = (
            f"[bold yellow]S:[/bold yellow] {question}\n\n"
            f"[bold green]C:[/bold green] {answer}"
        )

        _console.print(
            Panel(
                body,
                title=f"[bold]Sorgu #{index}[/bold]",
                border_style="blue",
                expand=False,
            )
        )
        _console.print()
else:
    print(
        f"\n--- {len(prompt_list)} prompt gönderildi, "
        f"{len(responses)} yanıt alındı ---\n"
    )

    for index, response in enumerate(responses, 1):
        print(f"[{index}] S: {prompt_list[index - 1]}")
        print(f"    C: {_extract_assistant(response)}")
        print()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                             Gemini Chat Target - Config                                                         │
│                                                                                                                 │
│   model_name              gemini-3.5-flash-lite                                                                 │
│   endpoint                https://generativelanguage.googleapis.com/v1beta/openai/                              │
│   max_completion_tokens   300                                                                                   │
│   temperature             0.7                                                                                   │
│   top_p                   0.95                                                                                  │
│   api_key                 GEMINI_API_KEY (env)                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The response was truncated because it reached the token limit (finish_reason='length'). Reasoning models consume tokens on hidden reasoning in addition to the visible answer, so a low max_completion_tokens can truncate or empty the response. Increase max_completion_tokens if you expected complete content.


────────────────────────────────────── Gönderilen 3 prompt - Alınan 3 yanıt ───────────────────────────────────────

╭─────────────────────────────────────────────────── Sorgu #1 ────────────────────────────────────────────────────╮
│ S: What is SQL injection in one sentence?                                                                       │
│                                                                                                                 │
│ C: SQL injection is a security vulnerability where an attacker can manipulate database queries by inserting     │
│ malicious SQL code into input fields, potentially allowing them to view, modify, or delete unauthorized data.   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Sorgu #2 ────────────────────────────────────────────────────╮
│ S: What is prompt injection in one sentence?                                                                    │
│                                                                                                                 │
│ C: Prompt injection is a security vulnerability where an attacker manipulates the input to a large language     │
│ model to bypass its guardrails and execute unintended instructions.                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────────── Sorgu #3 ────────────────────────────────────────────────────╮
│ S: What is the role of Data Science?                                                                            │
│                                                                                                                 │
│ C: At its core, the role of **Data Science** is to extract actionable meaning, patterns, and value from vast    │
│ amounts of data. In today’s digital world, organizations generate massive quantities of data every second (from │
│ user clicks and social media posts to financial transactions and IoT sensor readings). Data science acts as the │
│ bridge between raw data and informed decision-making.                                                           │
│                                                                                                                 │
│ Here is a breakdown of the primary roles and functions of Data Science across different domains:                │
│                                                                                                                 │
│ ---                                                                                                             │
│                                                                                                                 │
│ ### 1. Turning Data into Actionable Insights                                                                    │
│ Raw data is usually messy and unstructured. Data scientists clean, process, and analyze this data to answer     │
│ complex business questions.                                                                                     │
│ * *Example:* An e-commerce company might use data science to figure out why customers are abandoning their      │
│ shopping carts at the last minute, leading to changes that improve the checkout process.                        │
│                                                                                                                 │
│ ### 2. Powering Predictive Modeling and Forecasting                                                             │
│ Data science doesn't just look at what happened in the past; it uses machine learning and statistical           │
│ algorithms to predict what will happen in the future.                                                           │
│ * *Example:* Predicting customer churn (who is likely to cancel a subscription), forecasting quarterly sales,   │
│ or predicting equipment failures in a factory before they happen (predictive maintenance).                      │
│                                                                                                                 │
│ ### 3. Enabling Personalization and Recommendation Systems                                                      │
│ Much of our daily digital experience is shaped by data science algorithms designed to tailor content to         │
│ individual preferences.                                                                                         │
│ * *Example:* Netflix, Spotify, and Amazon use data science models to analyze your                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [3]:
import json
import os
from pyrit.prompt_target import HTTPTarget
from pyrit.prompt_normalizer import PromptNormalizer
from pyrit.models import Message, MessagePiece

# --- Pretty print helpers ---
try:
    from rich.console import Console
    from rich.table import Table
    from rich.panel import Panel
    from rich.rule import Rule
    from rich import box
    _console = Console()
    _RICH = True
except ImportError:
    _RICH = False


# HTTPTarget lets PyRIT talk to ANY REST API.
# The request string looks exactly like what you'd copy out of Burp Suite.
# PyRIT replaces the {PROMPT} placeholder in the body with the actual prompt before sending.

GROQ_KEY = os.environ["GROQ_API_KEY"]
GROQ_MODEL = "openai/gpt-oss-120b"   # GPT-OSS 120B hosted on Groq

# Build the JSON body — {PROMPT} will be replaced by PyRIT
body = json.dumps({
    "model": GROQ_MODEL,
    "messages": [{"role": "user", "content": "{PROMPT}"}],
    "max_tokens": 150,
})

# Raw HTTP request string (Host + headers + blank line + body)
http_request = (
    f"POST /openai/v1/chat/completions HTTP/1.1\n"
    f"Host: api.groq.com\n"
    f"Authorization: Bearer {GROQ_KEY}\n"
    f"Content-Type: application/json\n"
    f"\n"
    f"{body}"
)

# callback_function receives the httpx Response object and returns the reply text
def extract_groq_text(response):
    return response.json()["choices"][0]["message"]["content"]


# HTTPTarget — how PyRIT talks to non-OpenAI-SDK APIs
http_target = HTTPTarget(
    http_request=http_request,
    callback_function=extract_groq_text,
)

# --- Config table ---
_config = [
    ("Method",       "POST"),
    ("Host",         "api.groq.com"),
    ("Path",         "/openai/v1/chat/completions"),
    ("Model",        GROQ_MODEL),
    ("max_tokens",   "150"),
    ("Placeholder",  "{PROMPT} (in body)"),
    ("Auth",         "GROQ_API_KEY (env)"),
]

if _RICH:
    cfg = Table(
        box=box.SIMPLE_HEAD,
        show_header=False,
        title="[bold cyan]HTTPTarget — Groq Config[/bold cyan]",
    )
    cfg.add_column("Setting", style="bold yellow", no_wrap=True)
    cfg.add_column("Value", style="white")
    for k, v in _config:
        cfg.add_row(k, str(v))
    _console.print()
    _console.print(Panel(cfg, border_style="cyan", title="[bold]Raw HTTP Target[/bold]"))
else:
    print("\n=== HTTPTarget — Groq Config ===")
    for k, v in _config:
        print(f"  {k:<12}: {v}")
    print()


# Send a real prompt through HTTPTarget
prompt_text = "What is prompt injection in one sentence?"

http_normalizer = PromptNormalizer()
http_msg = Message(message_pieces=[
    MessagePiece(role="user", original_value=prompt_text)
])

http_response = await http_normalizer.send_prompt_async(
    message=http_msg,
    target=http_target,
)


# --- Print the response in an enterprise format ---
def _extract_assistant(resp):
    for piece in resp.message_pieces:
        if piece.role == "assistant":
            return str(piece.original_value)
    return "<no response>"


answer = _extract_assistant(http_response)

if _RICH:
    _console.print()
    _console.print(Rule("[bold green]Groq · GPT-OSS 120B · HTTPTarget response[/bold green]"))
    _console.print()
    body_md = f"[bold yellow]Q:[/bold yellow] {prompt_text}\n\n[bold green]A:[/bold green] {answer}"
    _console.print(Panel(body_md, title="[bold]Query[/bold]", border_style="blue", expand=False))
    _console.print()
else:
    print("Response from Groq via HTTPTarget (raw HTTP):")
    print("-" * 50)
    print(answer)
    print()

╭──────────────────────────────────────────────── Raw HTTP Target ────────────────────────────────────────────────╮
│           HTTPTarget — Groq Config                                                                              │
│                                                                                                                 │
│   Method        POST                                                                                            │
│   Host          api.groq.com                                                                                    │
│   Path          /openai/v1/chat/completions                                                                     │
│   Model         openai/gpt-oss-120b                                                                             │
│   max_tokens    150                                                                                             │
│   Placeholder   {PROMPT} (in body)                                                                              │
│   Auth          GROQ_API_KEY (env)                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

──────────────────────────────────── Groq · GPT-OSS 120B · HTTPTarget response ────────────────────────────────────

╭───────────────────────────────────────────────────── Query ─────────────────────────────────────────────────────╮
│ Q: What is prompt injection in one sentence?                                                                    │
│                                                                                                                 │
│ A: Prompt injection is a technique where a user manipulates a language model’s input to make it produce         │
│ unintended or harmful outputs by overriding or steering its original instructions.                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [4]:
from pyrit.score import SubStringScorer

# --- Pretty print helpers ---
try:
    from rich.console import Console
    from rich.table import Table
    from rich.panel import Panel
    from rich.rule import Rule
    from rich import box
    _console = Console()
    _RICH = True
except ImportError:
    _RICH = False


SEARCH_SUBSTRING = "sorry"

# PyRIT 0.14: parameter is 'categories' (list), not 'category' (str)
refusal_scorer = SubStringScorer(
    substring=SEARCH_SUBSTRING,
    categories=["refusal"],
)

test_texts = [
    "I'm sorry, I can't help with that request as it violates my guidelines.",
    "Here is a detailed step-by-step explanation of how to do that.",
    "I'm sorry but I need to decline this particular task.",
    "Sure! Machine learning works by training models on data.",
]


# --- Header ---
if _RICH:
    _console.print()
    _console.print(Panel(
        f"[bold]SubStringScorer Demo[/bold]\n"
        f"[dim]Searching for substring:[/dim] [bold yellow]'{SEARCH_SUBSTRING}'[/bold yellow]\n"
        f"[dim]Category:[/dim] [bold]refusal[/bold]",
        title="[bold cyan]PyRIT · SubStringScorer[/bold cyan]",
        border_style="cyan",
        expand=False,
    ))
    _console.print()
else:
    print("=== SubStringScorer Demo ===")
    print(f"Checking for substring: '{SEARCH_SUBSTRING}'")
    print("(True = contains 'sorry', False = does not)")
    print("-" * 60)


# --- Score each text ---
rows = []
for text in test_texts:
    scores = await refusal_scorer.score_text_async(text=text)
    matched = scores[0].get_value()
    rows.append((matched, text))


# --- Render results ---
if _RICH:
    table = Table(
        box=box.ROUNDED,
        show_header=True,
        header_style="bold white on blue",
        title="[bold cyan]Scoring Results[/bold cyan]",
    )
    table.add_column("#", justify="right", style="dim", width=3)
    table.add_column("Match", justify="center", width=10)
    table.add_column("Text", style="white")

    for i, (matched, text) in enumerate(rows, 1):
        badge = "[bold green]✓ MATCH[/bold green]" if matched else "[bold red]✗ NO MATCH[/bold red]"
        table.add_row(str(i), badge, text)

    _console.print(table)

    matched_count = sum(1 for m, _ in rows if m)
    _console.print()
    _console.print(Rule(
        f"[bold]{matched_count} / {len(rows)} matched[/bold]"
    ))
    _console.print()
else:
    for i, (matched, text) in enumerate(rows, 1):
        icon = "MATCH" if matched else "NO MATCH"
        print(f"[{icon}] {text[:70]}")
        print()

    matched_count = sum(1 for m, _ in rows if m)
    print(f"Summary: {matched_count} / {len(rows)} matched")
    print()

╭──── PyRIT · SubStringScorer ─────╮
│ SubStringScorer Demo             │
│ Searching for substring: 'sorry' │
│ Category: refusal                │
╰──────────────────────────────────╯

                                       Scoring Results                                        
╭─────┬────────────┬─────────────────────────────────────────────────────────────────────────╮
│   # │   Match    │ Text                                                                    │
├─────┼────────────┼─────────────────────────────────────────────────────────────────────────┤
│   1 │  ✓ MATCH   │ I'm sorry, I can't help with that request as it violates my guidelines. │
│   2 │ ✗ NO MATCH │ Here is a detailed step-by-step explanation of how to do that.          │
│   3 │  ✓ MATCH   │ I'm sorry but I need to decline this particular task.                   │
│   4 │ ✗ NO MATCH │ Sure! Machine learning works by training models on data.                │
╰─────┴────────────┴─────────────────────────────────────────────────────────────────────────╯

────────────────────────────────────────────────── 2 / 4 matched ──────────────────────────────────────────────────